# 06: NYC Isochrones

Transit+walk r5py isochrones for Sports Illustrated Stadium vs. Etihad Park.

Uses `speed_walking=5.0` km/h rather than r5py's default.

**Outputs:** `data/processed/nyc_{isochrones_90min.geojson,tract_travel_times_90min.csv}`.


In [1]:
import os
from datetime import datetime, timedelta

import geopandas as gpd
import pandas as pd
import r5py

DATA_PROCESSED_DIR = os.path.join("..", "data", "processed")
OSM_CLIPPED = os.path.join("..", "data", "raw", "osm", "nyc-metro-clipped.osm.pbf")
# GTFS/census/processed inputs below are all vendored locally (data/raw, data/processed)
TRANSPORTATION_GTFS = os.path.join("..", "data", "raw", "gtfs")
TRANSPORTATION_CENSUS = os.path.join("..", "data", "raw", "census")
TRANSPORTATION_PROCESSED = os.path.join("..", "data", "processed")

MAP_BOUNDS = (-74.35, 40.60, -73.65, 40.90)
TRANSIT_THRESHOLDS = [30, 45, 60, 90]

# r5py's default speed_walking is 3.6 km/h (~2.2 mph) which is extremely low, bumped it to 5.0 km/hr
SPEED_WALKING_KMH = 5.0

## Departure scenario and tract-centroid destinations

Isochrones and the travel-time matrix route to the same set of Census tract centroids and use
the same departure time ("next Saturday, 6pm" -- a representative game-day scenario), so this
is computed once.


In [2]:
today = datetime.now()
days_until_saturday = (5 - today.weekday()) % 7
days_until_saturday = days_until_saturday if days_until_saturday > 0 else 7
next_saturday = today + timedelta(days=days_until_saturday)
departure = next_saturday.replace(hour=18, minute=0, second=0, microsecond=0)
print(f"Departure scenario: {departure.strftime('%A %Y-%m-%d %H:%M')}")

Departure scenario: Saturday 2026-08-29 18:00


In [3]:
tract_frames = []
for zip_name in ["tl_2023_36_tract.zip", "tl_2023_34_tract.zip"]:
    zip_path = os.path.join(TRANSPORTATION_CENSUS, zip_name)
    tracts = gpd.read_file(f"zip://{zip_path}")
    tract_frames.append(tracts)
tracts = pd.concat(tract_frames, ignore_index=True)
tracts = gpd.GeoDataFrame(tracts, crs=tract_frames[0].crs).to_crs("EPSG:4326")
tracts = tracts.cx[MAP_BOUNDS[0]:MAP_BOUNDS[2], MAP_BOUNDS[1]:MAP_BOUNDS[3]]
print(f"{len(tracts)} tracts within the metro bbox")

utm_crs = tracts.estimate_utm_crs()
tract_centroids = tracts.copy()
tract_centroids["geometry"] = tract_centroids.geometry.to_crs(utm_crs).centroid.to_crs("EPSG:4326")
destinations = tract_centroids[["GEOID", "geometry"]].rename(columns={"GEOID": "id"})
destinations["id"] = destinations["id"].astype(str)

2980 tracts within the metro bbox


## Transit+walk isochrones (Sports Illustrated Stadium vs. Etihad Park)


In [4]:
import glob

gtfs_feeds = sorted(glob.glob(os.path.join(TRANSPORTATION_GTFS, "*.zip")))
print(f"GTFS feeds ({len(gtfs_feeds)}): {[os.path.basename(f) for f in gtfs_feeds]}")
print("Building transit+walk transport network...")
transit_network = r5py.TransportNetwork(OSM_CLIPPED, gtfs_feeds, allow_errors=True)
print("Transport network built.")

stadiums = pd.read_csv(os.path.join(TRANSPORTATION_PROCESSED, "relocation_stadiums.csv"))
transit_sites = [
    {"id": row["stadium_id"], "label": row["label"], "name": row["name"],
     "lat": row["latitude"], "lon": row["longitude"]}
    for _, row in stadiums.iterrows()
]
transit_origins = gpd.GeoDataFrame(
    transit_sites, geometry=gpd.points_from_xy([s["lon"] for s in transit_sites], [s["lat"] for s in transit_sites]),
    crs="EPSG:4326",
)

GTFS feeds (29): ['caltrain.zip', 'cta.zip', 'gotriangle.zip', 'houston_metro.zip', 'kcata.zip', 'king_county_metro.zip', 'lametro_bus.zip', 'lynx.zip', 'metra.zip', 'mta_bus_bronx.zip', 'mta_bus_brooklyn.zip', 'mta_bus_company.zip', 'mta_bus_manhattan.zip', 'mta_bus_queens.zip', 'mta_bus_staten_island.zip', 'mta_lirr.zip', 'mta_subway.zip', 'njtransit_bus.zip', 'njtransit_rail.zip', 'path.zip', 'pierce_transit.zip', 'sdmts.zip', 'sound_transit_rail.zip', 'tarc.zip', 'trimet.zip', 'uta.zip', 'vta.zip', 'wmata_bus.zip', 'wmata_rail.zip']
Building transit+walk transport network...


/opt/anaconda3/lib/python3.13/site-packages/r5py/r5/transport_network.py:137: RuntimeWarning: R5 reported the following issues with GTFS file caltrain.zip: 
EmptyTableError: frequencies line 0: Table is present in zip file, but it has no entries.
  warnings.warn(


/opt/anaconda3/lib/python3.13/site-packages/r5py/r5/transport_network.py:137: RuntimeWarning: R5 reported the following issues with GTFS file cta.zip: 
EmptyTableError: frequencies line 0: Table is present in zip file, but it has no entries.
  warnings.warn(


/opt/anaconda3/lib/python3.13/site-packages/r5py/r5/transport_network.py:137: RuntimeWarning: R5 reported the following issues with GTFS file gotriangle.zip: 
EmptyTableError: frequencies line 0: Table is present in zip file, but it has no entries.
  warnings.warn(


/opt/anaconda3/lib/python3.13/site-packages/r5py/r5/transport_network.py:137: RuntimeWarning: R5 reported the following issues with GTFS file sound_transit_rail.zip: 
EmptyFieldError: transfers line 5, field 'transfer_type': No value supplied for a required column.
- EmptyFieldError: transfers line 2, field 'transfer_type': No value supplied for a required column.
- EmptyFieldError: transfers line 3, field 'transfer_type': No value supplied for a required column.
- EmptyFieldError: transfers line 4, field 'transfer_type': No value supplied for a required column.
  warnings.warn(


/opt/anaconda3/lib/python3.13/site-packages/r5py/r5/transport_network.py:137: RuntimeWarning: R5 reported the following issues with GTFS file tarc.zip: 
EmptyTableError: fare_attributes line 0: Table is present in zip file, but it has no entries.
- EmptyTableError: transfers line 0: Table is present in zip file, but it has no entries.
- EmptyTableError: frequencies line 0: Table is present in zip file, but it has no entries.
  warnings.warn(


Transport network built.


In [5]:
transit_isochrone_frames = []
for _, origin_row in transit_origins.iterrows():
    print(f"Computing isochrones for {origin_row['name']}...")
    iso = r5py.Isochrones(
        transit_network, origins=origin_row.geometry, isochrones=TRANSIT_THRESHOLDS,
        point_grid_resolution=250, transport_modes=[r5py.TransportMode.TRANSIT],
        access_modes=[r5py.TransportMode.WALK], egress_modes=[r5py.TransportMode.WALK],
        departure=departure, speed_walking=SPEED_WALKING_KMH,
    )
    iso["site_id"] = origin_row["id"]
    iso["site_label"] = origin_row["label"]
    iso["site_name"] = origin_row["name"]
    transit_isochrone_frames.append(iso)

transit_isochrones = gpd.GeoDataFrame(pd.concat(transit_isochrone_frames, ignore_index=True), crs=transit_isochrone_frames[0].crs)
transit_isochrones["travel_time_minutes"] = transit_isochrones["travel_time"].dt.total_seconds() / 60
transit_isochrones = transit_isochrones.drop(columns=["travel_time"])
transit_iso_path = os.path.join(DATA_PROCESSED_DIR, "nyc_isochrones_90min.geojson")
transit_isochrones.to_file(transit_iso_path, driver="GeoJSON")
print(f"Saved {len(transit_isochrones)} isochrone boundary lines to {transit_iso_path}")

Computing isochrones for Sports Illustrated Stadium...


Computing isochrones for Etihad Park...


Saved 8 isochrone boundary lines to ../data/processed/nyc_isochrones_90min.geojson


In [6]:
transit_matrix_frames = []
for _, origin_row in transit_origins.iterrows():
    print(f"Computing travel times from {origin_row['name']} to {len(destinations)} tract centroids...")
    origin_gdf = gpd.GeoDataFrame([{"id": origin_row["id"]}], geometry=[origin_row.geometry], crs="EPSG:4326")
    matrix = r5py.TravelTimeMatrix(
        transit_network, origins=origin_gdf, destinations=destinations,
        transport_modes=[r5py.TransportMode.TRANSIT],
        access_modes=[r5py.TransportMode.WALK], egress_modes=[r5py.TransportMode.WALK],
        departure=departure, speed_walking=SPEED_WALKING_KMH,
    )
    matrix = matrix.rename(columns={"to_id": "GEOID", "travel_time": "travel_time_minutes"})
    matrix["GEOID"] = matrix["GEOID"].astype(str)
    matrix["site_id"] = origin_row["id"]
    matrix["site_label"] = origin_row["label"]
    matrix["site_name"] = origin_row["name"]
    transit_matrix_frames.append(matrix)

transit_travel_times = pd.concat(transit_matrix_frames, ignore_index=True)
transit_tt_path = os.path.join(DATA_PROCESSED_DIR, "nyc_tract_travel_times_90min.csv")
transit_travel_times.to_csv(transit_tt_path, index=False)
print(f"Saved {len(transit_travel_times)} origin-tract travel times to {transit_tt_path}")
print("DONE")

Computing travel times from Sports Illustrated Stadium to 2980 tract centroids...


Computing travel times from Etihad Park to 2980 tract centroids...


Saved 5960 origin-tract travel times to ../data/processed/nyc_tract_travel_times_90min.csv
DONE
